# Spark: four small examples

Run the cells from top to bottom. Everything uses tiny local data; no socket server or cloud account is needed.

1. One batch, two parallel partitions.
2. Combining a batch price RDD with a DStream of sales.
3. The same DataFrame grouping on a finished dataset and a stream.
4. Stop and restart both streaming models using checkpoints.

Use **Start Spark Notebook.cmd** in this folder, then choose **Run → Run All Cells**.
The notebook includes saved results from a successful run. The examples take a few minutes.

## Setup

This cell selects the installed local runtime. `local[2]` gives Spark two worker threads on this computer.
We use Spark 3.5.7 because it supports both APIs used in the HTML.

In [1]:
import os
import sys
from pathlib import Path

project = Path.cwd()
if project.name == "outputs":
    project = project.parent

runtime = project / "work" / "spark-runtime"
if os.name == "nt" and runtime.exists():
    java = next((runtime / "java").iterdir())
    hadoop = runtime / "hadoop"
    os.environ["JAVA_HOME"] = str(java)
    os.environ["HADOOP_HOME"] = str(hadoop)
    os.environ["PATH"] = str(java / "bin") + os.pathsep + str(hadoop / "bin") + os.pathsep + os.environ["PATH"]

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

In [2]:
from pyspark.sql import SparkSession
from pyspark.streaming import StreamingContext

spark = (SparkSession.builder
         .master("local[2]")
         .appName("FourSimpleExamples")
         .config("spark.ui.enabled", "false")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.sql.shuffle.partitions", "2")
         .getOrCreate())

sc = spark.sparkContext
sc.setLogLevel("ERROR")
print("Spark", spark.version, "with", sc.defaultParallelism, "worker threads")

Spark 3.5.7 with 2 worker threads


### Small helpers for local files

Each run gets a new temporary folder, so **Run All** starts fresh. Within example 4, the checkpoint paths stay the same across each stop/restart.
`add_file` moves a finished file into the input folder. Spark only sees complete files.
`wait_until` lets an example wait for its result instead of guessing how fast your computer is.

In [3]:
import tempfile
import time

demo = Path(tempfile.mkdtemp(prefix="spark_examples_"))

def folder(name):
    path = demo / name
    path.mkdir()
    return path

def add_file(directory, name, text):
    ready = demo / ("ready_" + name)
    ready.write_text(text, encoding="utf-8")
    ready.replace(directory / name)

def wait_until(test):
    end = time.time() + 120
    while time.time() < end:
        if test():
            return
        time.sleep(0.5)
    raise TimeoutError("Spark did not produce the expected result within 120 seconds")

## 1. One batch can use parallel partitions

These six numbers represent **one small batch**. We put them into **two partitions**.
The short pause makes it easy to see the two tasks overlap. It is only a teaching aid.

This uses the RDD that could hold one micro-batch; the list itself is not a live stream.

In [4]:
batch = sc.parallelize([1, 2, 3, 4, 5, 6], 2)
print("Partitions:", batch.glom().collect())

def inspect_partition(number, records):
    import os
    import time
    values = list(records)
    start = time.time()
    time.sleep(2)
    end = time.time()
    return [(number, values, os.getpid(), start, end)]

tasks = batch.mapPartitionsWithIndex(inspect_partition).collect()
first_start = min(task[3] for task in tasks)

for number, values, worker, start, end in tasks:
    print(f"Partition {number}: {values}, worker {worker}, "
          f"start {start - first_start:.2f}s, end {end - first_start:.2f}s")

overlap = max(task[3] for task in tasks) < min(task[4] for task in tasks)
print("Tasks overlapped:", overlap)

Partitions: [[1, 2, 3], [4, 5, 6]]


Partition 0: [1, 2, 3], worker 18528, start 0.66s, end 2.66s
Partition 1: [4, 5, 6], worker 7272, start 0.00s, end 2.00s
Tasks overlapped: True


The records belong to one batch, but two tasks process different partitions at the same time.
`glom()` only collects the partition contents so we can inspect them.

**Batching chooses which records go together. Parallel execution shares the work.**
This local demonstration uses one computer; a cluster can place partitions on different computers.

## 2. Combining batch data with a DStream

A shop has a finished **batch dataset of prices** and a **stream of sales quantities**.
For each arriving batch of sales, calculate the money earned: quantity × price.

The prices are an **RDD**. The sales are a **DStream**. To join them, we use
`transform`: it gives us the RDD for one arriving batch, which we can join to the price RDD.

In [5]:
# Batch data: (product, price).
prices = sc.parallelize([("apple", 2), ("banana", 1)])

# Streaming data: (product, quantity sold).
ssc = StreamingContext(sc, 5)
sales = ssc.queueStream([sc.parallelize([("apple", 3), ("banana", 2)])])

# Switch from the DStream to each batch's RDD to combine the two datasets.
totals = sales.transform(lambda batch_rdd:
    batch_rdd.join(prices).mapValues(lambda pair: pair[0] * pair[1]))

# Collect the result of this tiny demonstration.
results = []
def collect_totals(batch_rdd):
    results.extend(batch_rdd.collect())

totals.foreachRDD(collect_totals)
ssc.start()
try:
    wait_until(lambda: len(results) == 2)
finally:
    ssc.stop(stopSparkContext=False, stopGraceFully=True)

print(sorted(results))

C:\Users\hoduc\Documents\Codex\2026-10-22\ew\work\spark-env\Lib\site-packages\pyspark\streaming\context.py:72: FutureWarning: DStream is deprecated as of Spark 3.4.0. Migrate to Structured Streaming.
  warnings.warn(


[('apple', 6), ('banana', 2)]


Expected result: `[('apple', 6), ('banana', 2)]`.
Three apples at 2 each earn 6; two bananas at 1 each earn 2.

The switch between abstractions is the key part:

- `sales` and `totals` are **DStreams**, representing successive batches.
- Inside `transform`, `batch_rdd` is one **RDD**. `join(prices)` combines two RDDs.
- `foreachRDD` switches to an RDD again to collect each batch's output.

The developer explicitly bridges the continuing stream and the finished batch dataset.
`queueStream` only supplies one test batch here, so no external sales server is needed.

## 3. One DataFrame grouping for batch and streaming

We use two input files. The first contains `cat dog cat`; the second will add `dog owl`.
`groupBy("word").count()` is the same calculation in both versions.

In [6]:
from pyspark.sql.functions import explode, split

input3 = folder("dataframe_input")
add_file(input3, "first.txt", "cat dog cat\n")

def words_from(lines):
    return lines.select(explode(split("value", " ")).alias("word"))

def group_words(words):
    return words.groupBy("word").count()

batch_lines = spark.read.text(str(input3))
batch_counts = group_words(words_from(batch_lines))
batch_counts.orderBy("word").show()

+----+-----+
|word|count|
+----+-----+
| cat|    2|
| dog|    1|
+----+-----+



In [7]:
stream_lines = spark.readStream.text(str(input3))
stream_words = words_from(stream_lines)
stream_counts = group_words(stream_words)

query = (stream_counts.writeStream
         .format("memory")
         .queryName("live_counts")
         .outputMode("complete")
         .start())

try:
    query.processAllAvailable()
    print("After the first file:")
    spark.table("live_counts").orderBy("word").show()

    add_file(input3, "second.txt", "dog owl\n")
    query.processAllAvailable()
    print("After the second file:")
    spark.table("live_counts").orderBy("word").show()
finally:
    query.stop()

After the first file:
+----+-----+
|word|count|
+----+-----+
| cat|    2|
| dog|    1|
+----+-----+



After the second file:
+----+-----+
|word|count|
+----+-----+
| cat|    2|
| dog|    2|
| owl|    1|
+----+-----+



The streaming result grows from `cat = 2, dog = 1` to `cat = 2, dog = 2, owl = 1`.
Spark keeps the counts as **state**. Complete mode publishes the whole current result.
The memory output is just for inspecting this example; example 4 uses files for restartable output.

### Why a final sort is different

A batch is finished, so `orderBy` can sort the whole dataset.
An input stream keeps receiving records. A global sort of those raw incoming rows in Append mode is rejected:

In [8]:
from pyspark.sql.utils import AnalysisException

try:
    bad_query = (stream_words.orderBy("word").writeStream
                 .format("memory")
                 .queryName("cannot_sort_forever")
                 .outputMode("append")
                 .start())
except AnalysisException:
    print("Expected rejection: Spark cannot globally sort an unbounded input in Append mode.")
else:
    bad_query.stop()
    raise AssertionError("This unsupported sort unexpectedly started")

Expected rejection: Spark cannot globally sort an unbounded input in Append mode.


Sorting a **current result** is different: Spark permits sorting after an aggregation in Complete mode.
In the working example we simply sorted the saved memory table for display. That is a finite snapshot, not a final ordering of all future input.
See the [Structured Streaming guide](https://spark.apache.org/docs/3.5.7/structured-streaming-programming-guide.html) for the supported combinations.

## 4. Checkpointing: save counts and continue

Both examples count the same two files, with a stop/restart between them.

| Moment | Expected counts |
|---|---|
| First file: `cat dog cat` | cat = 2, dog = 1 |
| Restart, then second file: `dog owl` | cat = 2, dog = 2, owl = 1 |

We deliberately keep the original input files and reuse each model's checkpoint directory.
If the first file were counted twice, `cat` would become 4. If state were lost, `cat` would disappear.
This demonstrates an orderly stop/restart, not a sudden machine failure. Local files are enough for this demonstration.

### 4a. DStream: stateful RDDs and a checkpointed context

`updateStateByKey` maintains the running counts. `ssc.checkpoint(...)` enables checkpointing.
`getOrCreate` restores the saved context, or calls our setup function if there is no checkpoint yet.

We use a file source here. The `queueStream` source in example 2 is **not suitable for checkpoint recovery**.

In [9]:
ds_input = folder("dstream_input")
ds_output = folder("dstream_output")
ds_checkpoint = demo / "dstream_checkpoint"

def update_count(new_counts, old_count):
    return sum(new_counts) + (old_count or 0)

def save_rdd(batch_time, rdd):
    name = str(int(batch_time.timestamp() * 1000))
    rdd.saveAsTextFile(str(ds_output / name))

def make_dstream():
    context = StreamingContext(sc, 10)
    context.checkpoint(str(ds_checkpoint))
    lines = context.textFileStream(str(ds_input))
    pairs = lines.flatMap(lambda line: line.split()).map(lambda word: (word, 1))
    totals = pairs.updateStateByKey(update_count)
    totals.checkpoint(10)
    totals.foreachRDD(save_rdd)
    return context

This small inspection helper reads the latest completed output folder. Spark writes `_SUCCESS` when that output finishes.
It lets us check the real counts written by the background stream.

In [10]:
import ast

def dstream_counts():
    snapshots = sorted(ds_output.iterdir(), key=lambda path: path.name, reverse=True)
    for snapshot in snapshots:
        if (snapshot / "_SUCCESS").exists():
            rows = []
            for part in snapshot.glob("part-*"):
                rows.extend(ast.literal_eval(line) for line in part.read_text().splitlines())
            return dict(rows)
    return {}

In [11]:
ssc = StreamingContext.getOrCreate(str(ds_checkpoint), make_dstream)
ssc.start()
try:
    add_file(ds_input, "first.txt", "cat dog cat\n")
    wait_until(lambda: dstream_counts() == {"cat": 2, "dog": 1})
    print("Before stopping:", sorted(dstream_counts().items()))
    time.sleep(4)  # Give the periodic checkpoint time to finish.
finally:
    ssc.stop(stopSparkContext=False, stopGraceFully=True)

Before stopping: [('cat', 2), ('dog', 1)]


In [12]:
# Same checkpoint path: restore the graph and state.
ssc = StreamingContext.getOrCreate(str(ds_checkpoint), make_dstream)
ssc.start()
try:
    add_file(ds_input, "second.txt", "dog owl\n")
    wait_until(lambda: dstream_counts() == {"cat": 2, "dog": 2, "owl": 1})
    print("After restarting:", sorted(dstream_counts().items()))
finally:
    ssc.stop(stopSparkContext=False, stopGraceFully=True)

print("DStream checkpoint files exist:", any(ds_checkpoint.iterdir()))

After restarting: [('cat', 2), ('dog', 2), ('owl', 1)]


DStream checkpoint files exist: True


The `cat = 2` state survived. DStreams combine RDD lineage/recomputation with saved state and context metadata.
Whether lost input can be read again depends on the source: a plain socket cannot replay everything like retained files can.
The [DStream guide](https://spark.apache.org/docs/3.5.7/streaming-programming-guide.html#checkpointing) explains this distinction.
The simple per-batch output above is not a general exactly-once sink for arbitrary crash/retry scenarios.

### 4b. Structured Streaming: query progress and managed state

The grouping expression is unchanged. The query now has a `checkpointLocation`.
It saves source progress, state, and batch commit information.

`foreachBatch` writes the full count table into a folder named with its batch ID.
Overwriting that same folder makes a retried batch replace its previous snapshot, rather than append duplicate copies.
This is a simple local demonstration; the output behavior is explicitly defined by our callback.

In [13]:
ss_input = folder("structured_input")
ss_output = folder("structured_output")
ss_checkpoint = demo / "structured_checkpoint"

def save_table(table, batch_id):
    table.coalesce(1).write.mode("overwrite").json(str(ss_output / str(batch_id)))

def start_structured():
    lines = spark.readStream.text(str(ss_input))
    counts = group_words(words_from(lines))
    return (counts.writeStream
            .outputMode("complete")
            .option("checkpointLocation", str(ss_checkpoint))
            .foreachBatch(save_table)
            .start())

In [14]:
import json

def structured_counts():
    snapshots = [path for path in ss_output.iterdir() if path.name.isdigit()]
    latest = max(snapshots, key=lambda path: int(path.name))
    counts = {}
    for part in latest.glob("part-*.json"):
        for line in part.read_text().splitlines():
            row = json.loads(line)
            counts[row["word"]] = row["count"]
    return counts

add_file(ss_input, "first.txt", "cat dog cat\n")
query = start_structured()
try:
    query.processAllAvailable()
    print("Before stopping:", sorted(structured_counts().items()))
    assert structured_counts() == {"cat": 2, "dog": 1}
finally:
    query.stop()

Before stopping: [('cat', 2), ('dog', 1)]


In [15]:
# Same query and checkpoint: read only new input and restore old counts.
add_file(ss_input, "second.txt", "dog owl\n")
query = start_structured()
try:
    query.processAllAvailable()
    print("After restarting:", sorted(structured_counts().items()))
    assert structured_counts() == {"cat": 2, "dog": 2, "owl": 1}
finally:
    query.stop()

for name in ["offsets", "commits", "state"]:
    print(name, "saved:", (ss_checkpoint / name).exists())

After restarting: [('cat', 2), ('dog', 2), ('owl', 1)]
offsets saved: True
commits saved: True
state saved: True


Both recovered `cat = 2, dog = 2, owl = 1`.

| DStreams | Structured Streaming |
|---|---|
| RDD operations plus `updateStateByKey` | DataFrame grouping with managed state |
| `ssc.checkpoint(path)` | `.option("checkpointLocation", path)` |
| `StreamingContext.getOrCreate(path, setup)` | Start the same query with the same checkpoint |
| RDD/state checkpoints and saved context metadata | Source offsets, state store, and query commit logs |

Checkpointing does not make every external action happen exactly once. Retained/replayable input and retry-safe output are also needed.
The notebook demonstrates restored counts and progress, not a universal delivery guarantee.

In [16]:
spark.stop()
print("Finished. All four examples ran.")

Finished. All four examples ran.
